In [1]:
!pip install -q langchain langchain-groq langgraph langchain-chroma langchain-huggingface sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [2]:
import langchain
import langchain_groq
import langgraph
import chromadb
import sentence_transformers

print("All required packages imported successfully!")

All required packages imported successfully!


In [3]:
from google.colab import userdata
from langchain_groq import ChatGroq

groq_api_key = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=groq_api_key,
    temperature=0
)

print("Groq connected successfully!")

Groq connected successfully!


In [4]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

memory = InMemorySaver()

SYSTEM = """
You are the AI assistant for Sweet Crumbs Home Bakery.

Confirmed menu:
- Chocolate Cake: Rs. 650
- Plum Cake: Rs. 500

Both cakes are available in eggless versions.

Do not invent prices, delivery details, business hours,
or other unconfirmed business policies.
If information is unavailable, say:
"I'll check with the owner."
"""

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM,
    checkpointer=memory
)

print("Conversation memory enabled successfully!")

Conversation memory enabled successfully!


In [5]:
def chat(thread_id, message):
    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    response = agent.invoke(
        {"messages": [{"role": "user", "content": message}]},
        config
    )

    return response["messages"][-1].content

print("Chat function created successfully!")

Chat function created successfully!


In [6]:
response = chat("ravi", "Hi, my name is Ravi. I want 2 Plum Cakes.")

print("Ravi:", response)

Ravi: Sure thing, Ravi!  
- 2 Plum Cakes (₹500 each) = ₹1,000

Both cakes are available in egg‑less versions if you’d like.  

Could you let me know if you’d prefer to pick them up in‑store or have them delivered? If delivery, I’ll need the address and a contact number.  

Let me know, and I’ll get everything ready for you!


In [7]:
SYSTEM = """
You are the AI assistant for Sweet Crumbs Home Bakery.

Confirmed menu:
- Chocolate Cake: Rs. 650
- Plum Cake: Rs. 500
- Both cakes have eggless versions.

Rules:
1. Remember each customer's conversation.
2. Never invent delivery, pickup, payment, address,
   contact, booking or business-hour details.
3. Never claim an order is confirmed or placed.
4. If information is unknown, say:
   "I'll check with the owner."
5. Be polite and concise.
"""

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM,
    checkpointer=memory
)

print("Agent instructions updated successfully!")

Agent instructions updated successfully!


In [8]:
response = chat("ravi", "Can you make them eggless?")

print("Ravi:", response)

Ravi: Absolutely, Ravi!  
I’ll note that you’d like both Plum Cakes to be eggless.

Could you let me know whether you’d like to pick them up in‑store or have them delivered? If delivery, I’ll need the address and a contact number.  

Thanks!


In [9]:
response = chat("sravani", "Hi! My name is Sravani. Do you make cupcakes?")

print("Sravani:", response)

Sravani: Hi Sravani! We currently offer chocolate and plum cakes (both eggless). We don’t have cupcakes at the moment. How can I help you today?


In [10]:
response = chat("ravi", "What cake and quantity did I ask for?")

print("Ravi:", response)


Ravi: You asked for **2 Plum Cakes** (eggless).


In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

sentences = [
    "Do you deliver cakes?",
    "Can you send a cake to my home?",
    "What is the price of chocolate cake?"
]

vectors = embeddings.embed_documents(sentences)

print("Number of sentences:", len(vectors))
print("Vector dimensions:", len(vectors[0]))
print("First sentence vector (first 10 values):", vectors[0][:10])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of sentences: 3
Vector dimensions: 384
First sentence vector (first 10 values): [0.008889565244317055, 0.04305553808808327, 0.017325514927506447, 0.028363311663269997, -0.061244454234838486, -0.03410574048757553, 0.05845814198255539, -0.1305638551712036, -0.0025406794156879187, -0.00881128292530775]


In [12]:
from langchain_chroma import Chroma

faq_documents = [
    "Chocolate Cake costs Rs. 650.",
    "Plum Cake costs Rs. 500.",
    "Both Chocolate Cake and Plum Cake are available in eggless versions.",
    "Delivery information is not available. Please check with the owner.",
    "Business hours are not available. Please check with the owner.",
    "Cupcake availability is not confirmed. Please check with the owner."
]

vector_store = Chroma.from_texts(
    texts=faq_documents,
    embedding=embeddings,
    collection_name="sweet_crumbs_faq"
)

print("ChromaDB vector database created successfully!")
print("Number of FAQ documents:", vector_store._collection.count())

ChromaDB vector database created successfully!
Number of FAQ documents: 6


In [13]:
query = "Can you send a cake to my house?"

results = vector_store.similarity_search(
    query,
    k=2
)

print("Customer question:", query)
print("\nRelevant information found:")

for i, doc in enumerate(results, start=1):
    print(f"{i}. {doc.page_content}")

Customer question: Can you send a cake to my house?

Relevant information found:
1. Chocolate Cake costs Rs. 650.
2. Both Chocolate Cake and Plum Cake are available in eggless versions.


In [14]:
query = "Can you send a cake to my house?"

results = vector_store.similarity_search(
    query,
    k=4
)

print("Customer question:", query)
print("\nRetrieved FAQ information:")

for i, doc in enumerate(results, start=1):
    print(f"{i}. {doc.page_content}")

Customer question: Can you send a cake to my house?

Retrieved FAQ information:
1. Chocolate Cake costs Rs. 650.
2. Both Chocolate Cake and Plum Cake are available in eggless versions.
3. Plum Cake costs Rs. 500.
4. Cupcake availability is not confirmed. Please check with the owner.


In [15]:
queries = [
    "What is your delivery policy?",
    "How much does a chocolate cake cost?",
    "Do you have eggless cakes?"
]

for query in queries:
    results = vector_store.similarity_search(query, k=2)

    print("\nCustomer:", query)
    for doc in results:
        print("-", doc.page_content)


Customer: What is your delivery policy?
- Delivery information is not available. Please check with the owner.
- Chocolate Cake costs Rs. 650.

Customer: How much does a chocolate cake cost?
- Chocolate Cake costs Rs. 650.
- Plum Cake costs Rs. 500.

Customer: Do you have eggless cakes?
- Both Chocolate Cake and Plum Cake are available in eggless versions.
- Cupcake availability is not confirmed. Please check with the owner.


In [16]:
from langchain.tools import tool

@tool
def search_faq(question: str) -> str:
    """Search Sweet Crumbs bakery FAQs and policies."""
    results = vector_store.similarity_search(question, k=3)

    if not results:
        return "Information not available. Please check with the owner."

    return "\n".join(
        doc.page_content for doc in results
    )

print("FAQ search tool created successfully!")

FAQ search tool created successfully!


In [17]:
agent = create_agent(
    model=llm,
    tools=[search_faq],
    system_prompt=SYSTEM,
    checkpointer=memory
)

print("RAG-enabled AI agent created successfully!")

RAG-enabled AI agent created successfully!


In [18]:
response = chat(
    "customer1",
    "Do you deliver cakes to my home?"
)

print("Customer: Do you deliver cakes to my home?")
print("Assistant:", response)

Customer: Do you deliver cakes to my home?
Assistant: I’m not sure about delivery options. I’ll check with the owner and get back to you.


In [19]:
response = chat(
    "customer2",
    "What is the price of a Plum Cake?"
)

print("Customer: What is the price of a Plum Cake?")
print("Assistant:", response)

Customer: What is the price of a Plum Cake?
Assistant: The price of a Plum Cake is Rs. 500.


In [20]:
response = chat(
    "ravi",
    "What cakes did I ask about earlier, and are they eggless?"
)

print("Customer: What cakes did I ask about earlier, and are they eggless?")
print("Assistant:", response)

Customer: What cakes did I ask about earlier, and are they eggless?
Assistant: You asked about **Plum Cakes** earlier, and you’d like them to be **eggless**.


In [21]:
tests = [
    ("customer3", "Do you sell cupcakes?"),
    ("customer4", "What is the price of Chocolate Cake?"),
    ("customer5", "Are both cakes available in eggless versions?")
]

for customer_id, question in tests:
    print("\nCustomer:", question)
    response = chat(customer_id, question)
    print("Assistant:", response)


Customer: Do you sell cupcakes?
Assistant: I’m sorry, but we don’t offer cupcakes.

Customer: What is the price of Chocolate Cake?
Assistant: The Chocolate Cake is priced at **Rs. 650**.

Customer: Are both cakes available in eggless versions?
Assistant: Yes, both the Chocolate Cake and Plum Cake are available in eggless versions.
